# Predicting Bounding Boxes (PyTorch)

Welcome to Course 3, Week 1 Programming Assignment!

In this week's assignment, you'll build a model to predict bounding boxes around images.  
- You will use transfer learning on any of the pre-trained models available in torchvision.
- You'll be using the [Caltech Birds - 2010](http://www.vision.caltech.edu/visipedia/CUB-200.html) dataset.

> This is a PyTorch port of the original TensorFlow/Keras assignment. The exercises are the same (build a MobileNetV2 feature extractor, dense layers and a bounding-box regression head, then train the model), but they are written as `nn.Module`s and an explicit training loop. The dataset is the TFRecord archive prepared for the course; it is read with the pure-Python `tfrecord` package, so TensorFlow is not needed. Note that the Coursera autograder expects a Keras model file, so the PyTorch model cannot be submitted for grading.

- [Initial steps](#0)
  - [0.1 Set up your environment](#0-1)
  - [0.2 Choose the device](#0-2)
  - [0.3 Imports](#0-3)
  - [0.4 Download and Extract the Dataset](#0-4)
- [1. Visualization Utilities](#1)
  - [1.1 Bounding Boxes Utilities](#1-1)
  - [1.2 Data and Predictions Utilities](#1-2)
- [2. Preprocessing and Loading the Dataset](#2)
  - [2.1 Preprocessing Utilities](#2-1)
  - [2.2 Visualize the prepared Data](#2-2)
  - [2.3 Loading the Dataset](#2-3)
- [3. Define the Network](#3)
  - [Exercise 1](#ex-01)
  - [Exercise 2](#ex-02)
  - [Exercise 3](#ex-03)
  - [Exercise 4](#ex-04)
  - [Exercise 5](#ex-05)
- [4. Training the Model](#4)
  - [Prepare to train the model](#4.1)
  - [Exercise 6](#ex-06)
  - [Fit the model to the data](#4.2)
  - [Exercise 7](#ex-07)
- [5. Validate the Model](#5)
  - [5.1 Loss](#5-1)
  - [5.2 Plot the Loss Function](#5-2)  
  - [5.3 Evaluate performance using IoU](#5-3)
- [6. Visualize Predictions](#6)
- [7. Save the Model](#7)

<a name="0"></a>
## 0. Initial steps

<a name="0-1"></a>
## 0.1 Set up your environment

- This notebook runs locally in the `cv-pytorch` environment created with `uv` (see the README). Make sure the notebook kernel is the one from that environment.

<a name="0-2"></a>
## 0.2 Choose the device
- Training this model on a CPU is slow. The notebook automatically uses a **GPU** (CUDA) or Apple's **MPS** backend when one is available.

<a name="0-3"></a>
## 0.3 Imports

In [ ]:
import os, re, time, json, zipfile, io, glob, math
import urllib.request
import PIL.Image, PIL.ImageFont, PIL.ImageDraw
import numpy as np
from matplotlib import pyplot as plt
import cv2

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights
from torchinfo import summary
from tfrecord.reader import tfrecord_loader

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)

<a name="0-4"></a>
## 0.4 Download and Extract the Dataset

In [ ]:
# Download the dataset (about 690 MB; skipped if it is already there)
os.makedirs("data", exist_ok=True)
if not os.path.exists("data/caltech_birds2010_011.zip"):
    urllib.request.urlretrieve("https://storage.googleapis.com/tensorflow-3-public/datasets/caltech_birds2010_011.zip",
                               "data/caltech_birds2010_011.zip")

In [ ]:
# Specify the data directory
data_dir = "./data"

# Extract the dataset into the data directory
if not os.path.exists(os.path.join(data_dir, "caltech_birds2010")):
    with zipfile.ZipFile('data/caltech_birds2010_011.zip') as zipref:
        zipref.extractall(data_dir)
print(os.listdir(os.path.join(data_dir, "caltech_birds2010", "0.1.1")))

<a name="1"></a>
## 1. Visualization Utilities

<a name="1-1"></a>
### 1.1 Bounding Boxes Utilities

We have provided you with some functions which you will use to draw bounding boxes around the birds in the `image`.

- `draw_bounding_box_on_image`: Draws a single bounding box on an image.
- `draw_bounding_boxes_on_image`: Draws multiple bounding boxes on an image.
- `draw_bounding_boxes_on_image_array`: Draws multiple bounding boxes on an array of images.

In [ ]:
def draw_bounding_box_on_image(image, ymin, xmin, ymax, xmax, color=(255, 0, 0), thickness=5):
    """
    Adds a bounding box to an image.
    Bounding box coordinates can be specified in either absolute (pixel) or
    normalized coordinates by setting the use_normalized_coordinates argument.

    Args:
      image: a numpy array (height, width, 3).
      ymin: ymin of bounding box.
      xmin: xmin of bounding box.
      ymax: ymax of bounding box.
      xmax: xmax of bounding box.
      color: color to draw bounding box. Default is red.
      thickness: line thickness. Default value is 4.
    """

    image_width = image.shape[1]
    image_height = image.shape[0]
    cv2.rectangle(image, (int(xmin), int(ymin)), (int(xmax), int(ymax)), color, thickness)


def draw_bounding_boxes_on_image(image, boxes, color=[], thickness=5):
    """
    Draws bounding boxes on image.

    Args:
      image: a numpy array (height, width, 3).
      boxes: a 2 dimensional numpy array of [N, 4]: (xmin, ymin, xmax, ymax) in pixels.
      color: color to draw bounding box. Default is red.
      thickness: line thickness. Default value is 4.

    Raises:
      ValueError: if boxes is not a [N, 4] array
    """

    boxes_shape = boxes.shape
    if not boxes_shape:
        return
    if len(boxes_shape) != 2 or boxes_shape[1] != 4:
        raise ValueError('Input must be of size [N, 4]')
    for i in range(boxes_shape[0]):
        draw_bounding_box_on_image(image, boxes[i, 1], boxes[i, 0], boxes[i, 3],
                                 boxes[i, 2], color[i], thickness)


def draw_bounding_boxes_on_image_array(image, boxes, color=[], thickness=5):
    '''
    Draws bounding boxes on image (numpy array).

    Args:
      image: a numpy array object.
      boxes: a 2 dimensional numpy array of [N, 4]: (xmin, ymin, xmax, ymax) in pixels.
      color: color to draw bounding box. Default is red.
      thickness: line thickness. Default value is 4.

    Returns:
      the same array, with the boxes drawn on it in place

    Raises:
      ValueError: if boxes is not a [N, 4] array
    '''

    draw_bounding_boxes_on_image(image, boxes, color, thickness)

    return image

<a name="1-2"></a>
### 1.2 Data and Predictions Utilities

We've given you some helper functions and code that are used to visualize the data and the model's predictions.

- `display_digits_with_boxes`: This displays a row of "digit" images along with the model's predictions for each image.
- `plot_metrics`: This plots a given metric (like loss) as it changes over multiple epochs of training.  

In [ ]:
# Matplotlib config
plt.rc('image', cmap='gray')
plt.rc('grid', linewidth=0)
plt.rc('xtick', top=False, bottom=False, labelsize='large')
plt.rc('ytick', left=False, right=False, labelsize='large')
plt.rc('axes', facecolor='F8F8F8', titlesize="large", edgecolor='white')
plt.rc('text', color='a8151a')
plt.rc('figure', facecolor='F0F0F0')# Matplotlib fonts
MATPLOTLIB_FONT_DIR = os.path.join(os.path.dirname(plt.__file__), "mpl-data/fonts/ttf")


# utility to display a row of digits with their predictions
def display_digits_with_boxes(images, pred_bboxes, bboxes, iou, title, bboxes_normalized=False, iou_threshold=0.5):

    '''
    Displays a row of images with their predicted and ground truth boxes.

    Predicted boxes are drawn red and ground truth boxes green. When an IoU is given it is
    printed under each image, in red if it falls below `iou_threshold`.

    Args:
      images (array) -- images to draw on
      pred_bboxes (array) -- predicted boxes, normalized to [0, 1]
      bboxes (array) -- ground truth boxes
      iou (array) -- IoU per image, or an empty array to omit
      title (string) -- title for the figure
      bboxes_normalized (bool) -- True if `bboxes` is normalized rather than in pixels
      iou_threshold (float) -- IoU below which the score is printed in red
    '''
    n = len(images)

    fig = plt.figure(figsize=(20, 4))
    plt.title(title)
    plt.yticks([])
    plt.xticks([])

    for i in range(n):
      ax = fig.add_subplot(1, 10, i+1)
      bboxes_to_plot = []
      if (len(pred_bboxes) > i):
        bbox = pred_bboxes[i]
        bbox = [bbox[0] * images[i].shape[1], bbox[1] * images[i].shape[0], bbox[2] * images[i].shape[1], bbox[3] * images[i].shape[0]]
        bboxes_to_plot.append(bbox)

      if (len(bboxes) > i):
        bbox = bboxes[i]
        if bboxes_normalized == True:
          bbox = [bbox[0] * images[i].shape[1],bbox[1] * images[i].shape[0], bbox[2] * images[i].shape[1], bbox[3] * images[i].shape[0] ]
        bboxes_to_plot.append(bbox)

      img_to_draw = draw_bounding_boxes_on_image_array(image=np.ascontiguousarray(images[i]), boxes=np.asarray(bboxes_to_plot), color=[(255,0,0), (0, 255, 0)])
      plt.xticks([])
      plt.yticks([])

      plt.imshow(img_to_draw)

      if len(iou) > i :
        color = "black"
        if (iou[i][0] < iou_threshold):
          color = "red"
        ax.text(0.2, -0.3, "iou: %s" %(iou[i][0]), color=color, transform=ax.transAxes)


# utility to display training and validation curves
def plot_metrics(history, metric_name, title, ylim=5):
    '''
    Plots a training metric and its validation counterpart against the epoch number.

    Args:
      history (dict) -- metric name to list of per-epoch values
      metric_name (string) -- key to plot, for example 'loss'
      title (string) -- title for the figure
      ylim (float) -- upper limit of the y axis
    '''
    plt.title(title)
    plt.ylim(0,ylim)
    plt.plot(history[metric_name],color='blue',label=metric_name)
    plt.plot(history['val_' + metric_name],color='green',label='val_' + metric_name)

<a name="2"></a>
## 2. Preprocess and Load the Dataset

<a name="2-1"></a>
### 2.1 Preprocessing Utilities

We have given you some helper functions to pre-process the image data.

#### read_image
- Resizes `image` to (224, 224)
- Normalizes `image` (the torchvision MobileNetV2 weights expect inputs scaled to `[0, 1]` and normalized with the ImageNet mean and standard deviation)
- Translates and normalizes bounding boxes

In [ ]:
normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

def read_image(image, bbox):
    '''
    Resizes an image to 224x224, normalizes it, and rescales its box to [0, 1].

    Args:
      image (PIL.Image) -- the original image
      bbox (list) -- [xmin, ymin, xmax, ymax] in pixels

    Returns:
      image tensor of shape (3, 224, 224), bounding box tensor [xmin, ymin, xmax, ymax] normalized to [0, 1]
    '''
    factor_x, factor_y = image.size   # (width, height)

    image = image.resize((224, 224), PIL.Image.BILINEAR)

    image = transforms.functional.to_tensor(image)   # (3, 224, 224) in [0, 1]
    image = normalize(image)

    bbox_list = [bbox[0] / factor_x ,
                 bbox[1] / factor_y,
                 bbox[2] / factor_x ,
                 bbox[3] / factor_y]

    return image, torch.tensor(bbox_list, dtype=torch.float32)

#### read_image_with_shape
This is very similar to `read_image` except it also keeps a copy of the original image (before pre-processing) and returns this as well.
- Makes a copy of the original image.
- Resizes `image` to (224, 224)
- Normalizes `image`
- Translates and normalizes bounding boxes

In [ ]:
def read_image_with_shape(image, bbox):
    '''
    Preprocesses an image while also keeping the untouched original.

    Args:
      image (PIL.Image) -- the original image
      bbox (list) -- [xmin, ymin, xmax, ymax] in pixels

    Returns:
      (array, tensor, tensor) -- the original image, the preprocessed image (3, 224, 224),
      and the bounding box normalized to [0, 1]
    '''
    original_image = np.array(image)

    image, bbox_list = read_image(image, bbox)

    return original_image, image, bbox_list

#### CaltechBirdsDataset

- This `Dataset` reads the images and bounding boxes from the TFRecord files of the dataset (this is what `tfds.load` did in the TensorFlow version).
- It also denormalizes the bounding boxes: TFDS stores them as `[ymin, xmin, ymax, xmax]` in the range `[0, 1]`, and the dataset returns them as `[xmin, ymin, xmax, ymax]` in pixels (this is what `read_image_tfds_with_original_bbox` did).
- When a `transform` is given (e.g. `read_image`), it is applied to every item.

In [ ]:
class CaltechBirdsDataset(Dataset):

    '''
    Caltech Birds 2010 images and bounding boxes, read straight from the TFRecord shards.

    Boxes are stored by TFDS as normalized (ymin, xmin, ymax, xmax); this dataset returns
    them as (xmin, ymin, xmax, ymax) in pixels, which is what the rest of the notebook expects.
    '''
    def __init__(self, split, data_dir, transform=None):
        '''
        Reads every record of the split into memory, keeping the JPEG bytes undecoded.

        Args:
          split (string) -- 'train' or 'test'
          data_dir (string) -- directory the dataset zip was extracted into
          transform (callable) -- optional function applied to (image, bbox)
        '''
        shards = sorted(glob.glob(os.path.join(data_dir, "caltech_birds2010", "0.1.1", f"caltech_birds2010-{split}.tfrecord-*")))
        assert shards, f"no TFRecord files found for split '{split}' in {data_dir}"

        # keep the encoded JPEG bytes in memory and decode them on demand
        self.images = []
        self.bboxes = []
        for shard in shards:
            for record in tfrecord_loader(shard, None, description={"image": "byte", "bbox": "float"}):
                self.images.append(record["image"])
                ymin, xmin, ymax, xmax = record["bbox"]
                self.bboxes.append((float(xmin), float(ymin), float(xmax), float(ymax)))
        self.transform = transform

    def __len__(self):
        '''
        Reports how many items this split holds.

        Returns:
          int -- number of images in this split
        '''
        return len(self.images)

    def __getitem__(self, idx):
        '''
        Decodes image `idx` and converts its box to pixel coordinates.

        Args:
          idx (int) -- index of the image to fetch

        Returns:
          (PIL.Image, list) -- the image and [xmin, ymin, xmax, ymax] in pixels,
          or whatever `transform` returns when one was given
        '''
        image = PIL.Image.open(io.BytesIO(self.images[idx])).convert("RGB")
        factor_x, factor_y = image.size
        xmin, ymin, xmax, ymax = self.bboxes[idx]

        bbox_list = [xmin * factor_x,
                     ymin * factor_y,
                     xmax * factor_x,
                     ymax * factor_y]

        if self.transform is not None:
            return self.transform(image, bbox_list)
        return image, bbox_list

#### dataset_to_numpy_util
This function converts a `dataset` into numpy arrays of images and boxes.
- This will be used when visualizing the images and their bounding boxes

In [ ]:
def dataset_to_numpy_util(dataset, N=0):
    '''
    Converts a dataset into numpy arrays of images and boxes, for visualization.

    Args:
      dataset (Dataset) -- dataset yielding (image, bbox) pairs
      N (int) -- how many random items to take, or 0 for all

    Returns:
      (array, array) -- images and their bounding boxes
    '''
    # take N random items from the dataset
    indexes = np.random.choice(len(dataset), size=N, replace=False) if N > 0 else range(len(dataset))

    ds_images, ds_bboxes = [], []
    for i in indexes:
        image, bbox = dataset[i]
        ds_images.append(np.array(image))
        ds_bboxes.append(np.array(bbox))

    return (np.array(ds_images, dtype='object'), np.array(ds_bboxes, dtype='object'))

#### dataset_to_numpy_with_original_bboxes_util

- This function converts a `dataset` into numpy arrays of
  - original images
  - resized and normalized images
  - bounding boxes
- This will be used for plotting the original images with true and predicted bounding boxes.

In [ ]:
def dataset_to_numpy_with_original_bboxes_util(dataset, N=0):

    '''
    Converts a dataset into original images, preprocessed images, and normalized boxes.

    Args:
      dataset (Dataset) -- dataset yielding (image, bbox) pairs
      N (int) -- how many leading items to take, or 0 for all

    Returns:
      (array, array, array) -- original images, preprocessed images, normalized boxes
    '''
    indexes = range(N) if N > 0 else range(len(dataset))

    ds_original_images, ds_images, ds_bboxes = [], [], []

    for i in indexes:
        image, bbox = dataset[i]
        original_image, image, bbox = read_image_with_shape(image, bbox)
        ds_images.append(image.numpy())
        ds_bboxes.append(bbox.numpy())
        ds_original_images.append(original_image)

    return np.array(ds_original_images, dtype='object'), np.array(ds_images, dtype='float32'), np.array(ds_bboxes, dtype='float32')

<a name="2-2"></a>
### 2.2 Visualize the images and their bounding box labels
Now you'll take a random sample of images from the training and validation sets and visualize them by plotting the corresponding bounding boxes.

Visualize the **training** images and their bounding box labels

In [ ]:
def get_visualization_training_dataset():
    '''
    Loads the training split untransformed, for plotting.

    Returns:
      CaltechBirdsDataset -- the training split, untransformed, for visualization
    '''
    visualization_training_dataset = CaltechBirdsDataset(split="train", data_dir=data_dir)
    print(f"training set: {len(visualization_training_dataset)} images")
    return visualization_training_dataset


visualization_training_dataset = get_visualization_training_dataset()


(visualization_training_images, visualization_training_bboxes) = dataset_to_numpy_util(visualization_training_dataset, N=10)
display_digits_with_boxes(np.array(visualization_training_images), np.array([]), np.array(visualization_training_bboxes), np.array([]), "training images and their bboxes")

Visualize the **validation** images and their bounding boxes

In [ ]:
def get_visualization_validation_dataset():
    '''
    Loads the validation split untransformed, for plotting.

    Returns:
      CaltechBirdsDataset -- the validation split, untransformed, for visualization
    '''
    visualization_validation_dataset = CaltechBirdsDataset(split="test", data_dir=data_dir)
    print(f"validation set: {len(visualization_validation_dataset)} images")
    return visualization_validation_dataset


visualization_validation_dataset = get_visualization_validation_dataset()

(visualization_validation_images, visualization_validation_bboxes) = dataset_to_numpy_util(visualization_validation_dataset, N=10)
display_digits_with_boxes(np.array(visualization_validation_images), np.array([]), np.array(visualization_validation_bboxes), np.array([]), "validation images and their bboxes")

<a name="2-3"></a>
### 2.3 Load and prepare the datasets for the model

These next two functions read and prepare the datasets that you'll feed to the model.
- They use `read_image` to resize, and normalize each image and its bounding box label.
- They perform shuffling and batching through a `DataLoader`.
- You'll use these functions to create `training_dataset` and `validation_dataset`, which you will give to the model that you're about to build.

In [ ]:
BATCH_SIZE = 64

class TransformedDataset(Dataset):
    '''applies `read_image` to the items of a CaltechBirdsDataset'''
    def __init__(self, dataset):
        '''
        Stores the dataset whose items get preprocessed on access.

        Args:
          dataset (Dataset) -- dataset yielding raw (image, bbox) pairs
        '''
        self.dataset = dataset
    def __len__(self):
        '''
        Reports how many items the wrapped dataset holds.

        Returns:
          int -- number of items in the wrapped dataset
        '''
        return len(self.dataset)
    def __getitem__(self, idx):
        '''
        Fetches item `idx` from the wrapped dataset and preprocesses it.

        Args:
          idx (int) -- index of the item to fetch

        Returns:
          (tensor, tensor) -- image resized and normalized to (3, 224, 224), and its box in [0, 1]
        '''
        image, bbox = self.dataset[idx]
        return read_image(image, bbox)


def get_training_dataset(dataset):
    '''
    Wraps the training split in preprocessing and shuffles it into batches.

    Args:
      dataset (Dataset) -- the raw training split

    Returns:
      DataLoader -- shuffled batches of preprocessed images and boxes
    '''
    dataset = TransformedDataset(dataset)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
    return loader

def get_validation_dataset(dataset):
    '''
    Wraps the validation split in preprocessing and groups it into batches.

    Args:
      dataset (Dataset) -- the raw validation split

    Returns:
      DataLoader -- batches of preprocessed images and boxes, in order
    '''
    dataset = TransformedDataset(dataset)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE)
    return loader

training_dataset = get_training_dataset(visualization_training_dataset)
validation_dataset = get_validation_dataset(visualization_validation_dataset)

<a name="3"></a>
## 3. Define the Network

Bounding box prediction is treated as a "regression" task, in that you want the model to output numerical values.

- You will be performing transfer learning with **MobileNet V2**.  The model architecture is available in torchvision.
- You'll also use pretrained `'imagenet'` weights as a starting point for further training.  These weights are also readily available
- You will choose to retrain all layers of **MobileNet V2** along with the final regression layers.

**Note:** For the following exercises, each part of the network is a function returning an `nn.Module`, and Exercise 4 combines them in a single `nn.Module` (this plays the role of the Keras Functional API).

<a name='ex-01'></a>
### Exercise 1

Please build a feature extractor using MobileNetV2.

- First, create an instance of the mobilenet version 2 model
  - Please check out the documentation for [mobilenet_v2](https://pytorch.org/vision/stable/models/generated/torchvision.models.mobilenet_v2.html)
  - Set the following parameter:
    - weights: Use the pre-trained ImageNet weights (`MobileNet_V2_Weights.IMAGENET1K_V1`). Input images have height and width of 224 by 224, and have red, green and blue channels.

- Next, keep only the convolutional part of the network: you do not want to keep the "top" classifier, since you will customize your model for the current task.
    - The torchvision model has two children: `features` (the convolutional layers, which output 1280 feature maps of size 7x7 for a 224x224 image) and `classifier`. The `features` module is your feature extractor.

**Note**: please use mobilenet_v2 and not mobilenet_v3

In [ ]:
def feature_extractor():
    '''
    Builds the MobileNetV2 feature extractor (Exercise 1).

    Returns:
      nn.Module -- the convolutional part of MobileNetV2, producing (N, 1280, 7, 7)
    '''
    ### START CODE HERE ###
    # Create a mobilenet version 2 model object with the pre-trained ImageNet weights
    mobilenet_model = mobilenet_v2(weights=MobileNet_V2_Weights.IMAGENET1K_V1)

    # keep the convolutional part of this model object to get a feature extractor.
    # `.features` is the convolutional stack; `.classifier` is the ImageNet head we drop.
    feature_extractor = mobilenet_model.features

    ### END CODE HERE ###

    # return the feature_extractor
    return feature_extractor

<a name='ex-02'></a>
### Exercise 2

Next, you'll define the dense layers to be used by your model.

You'll be using the following layers
- [AdaptiveAvgPool2d](https://pytorch.org/docs/stable/generated/torch.nn.AdaptiveAvgPool2d.html) with an output size of 1: pools the `features` (this is the equivalent of Keras' `GlobalAveragePooling2D`).
- [Flatten](https://pytorch.org/docs/stable/generated/torch.nn.Flatten.html): flattens the pooled layer.
- [Linear](https://pytorch.org/docs/stable/generated/torch.nn.Linear.html): Add two fully connected layers:
    - A linear layer with 1024 neurons followed by a [ReLU](https://pytorch.org/docs/stable/generated/torch.nn.ReLU.html) activation.
    - A linear layer following that with 512 neurons followed by a ReLU activation.

**Note**: Remember that `Linear` layers need their number of input features: the MobileNetV2 features have 1280 channels.

In [ ]:
def dense_layers():
    '''
    Builds the dense layers that follow the feature extractor (Exercise 2).

    Returns:
      nn.Module -- pooling, flattening and two ReLU layers, ending in 512 features
    '''
    ### START CODE HERE ###
    layers = nn.Sequential(
        # global average pooling layer: (N, 1280, 7, 7) -> (N, 1280, 1, 1)
        nn.AdaptiveAvgPool2d(1),

        # flatten layer: (N, 1280, 1, 1) -> (N, 1280)
        nn.Flatten(),

        # 1024 linear layer, with relu
        nn.Linear(1280, 1024),
        nn.ReLU(),

        # 512 linear layer, with relu
        nn.Linear(1024, 512),
        nn.ReLU(),
    )

    ### END CODE HERE ###

    return layers

<a name='ex-03'></a>
### Exercise 3


Now you'll define a layer that outputs the bounding box predictions.
- You'll use a [Linear](https://pytorch.org/docs/stable/generated/torch.nn.Linear.html) layer.
- Remember that you have _4 units_ in the output layer, corresponding to (xmin, ymin, xmax, ymax).
- The prediction layer follows the previous dense layer, which has 512 outputs.

In [ ]:
def bounding_box_regression():
    '''
    Builds the layer that predicts the bounding box (Exercise 3).

    Returns:
      nn.Module -- layer mapping 512 features to 4 outputs (xmin, ymin, xmax, ymax)
    '''
    ### START CODE HERE ###
    # Linear layer with 4 outputs, one per box coordinate
    bounding_box_regression_output = nn.Linear(512, 4)

    ### END CODE HERE ###


    return bounding_box_regression_output

<a name='ex-04'></a>
### Exercise 4

Now, you'll use those functions that you have just defined above to construct the model.
- feature_extractor()
- dense_layers()
- bounding_box_regression()

Then you'll define the model as an [nn.Module](https://pytorch.org/docs/stable/generated/torch.nn.Module.html):
- create the three parts in `__init__` (store them as attributes; the bounding box layer must be stored in an attribute named `bounding_box`)
- chain them in `forward`

In [ ]:
class FinalModel(nn.Module):
    '''
    Bounding box regressor: MobileNetV2 features, dense layers, then four box outputs (Exercise 4).
    '''
    def __init__(self):
        '''
        Builds the feature extractor, the dense layers and the bounding box layer.
        '''
        super().__init__()
        ### START CODE HERE ###

        # features
        self.feature_cnn = feature_extractor()

        # dense layers
        self.dense = dense_layers()

        # bounding box
        self.bounding_box = bounding_box_regression()

        ### END CODE HERE ###

    def forward(self, inputs):
        '''
        Runs the image through the features, the dense layers, then the box head.

        Args:
          inputs (tensor) -- batch of images, shape (N, 3, 224, 224)

        Returns:
          tensor -- predicted boxes, shape (N, 4)
        '''
        ### START CODE HERE ###
        # features                          shape: (N, 3, 224, 224) -> (N, 1280, 7, 7)
        feature_cnn = self.feature_cnn(inputs)

        # dense layers                      shape: (N, 1280, 7, 7) -> (N, 512)
        last_dense_layer = self.dense(feature_cnn)

        # bounding box                      shape: (N, 512) -> (N, 4)
        bounding_box_output = self.bounding_box(last_dense_layer)

        ### END CODE HERE ###

        return bounding_box_output

<a name='ex-05'></a>
### Exercise 5

Define the model, move it to the device, and then define the optimizer and the loss (this is what `compile` did in Keras).
- model: use the `FinalModel` class that you just defined to create the model, and move it to `device` with `.to(device)`.
- optimizer: Set the optimizer to Stochastic Gradient Descent using [SGD](https://pytorch.org/docs/stable/generated/torch.optim.SGD.html)
    - When using SGD, set the `momentum` to 0.9 and use a learning rate of 0.01 (Keras' default learning rate).
- loss: Set the loss function to mean squared error ([MSELoss](https://pytorch.org/docs/stable/generated/torch.nn.MSELoss.html)).

In [ ]:
def define_and_compile_model(device):
    '''
    Creates the model, optimizer and loss function (Exercise 5).

    Args:
      device (torch.device) -- device the model is moved to

    Returns:
      (nn.Module, Optimizer, callable) -- the model on the chosen device, its SGD
      optimizer, and the mean squared error loss
    '''
    ### START CODE HERE ###
    # create the model and move it to the device
    model = FinalModel().to(device)

    # define the optimizer
    optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

    # define the loss function. Bounding box prediction is a regression task,
    # so the error is the squared distance between predicted and true coordinates.
    loss_fn = nn.MSELoss()

    ### END CODE HERE ###


    return model, optimizer, loss_fn

Run the cell below to define your model and print the model summary.

In [ ]:
# define your model
model, optimizer, loss_fn = define_and_compile_model(device)
# print model layers
summary(model, input_size=(1, 3, 224, 224), device=device, depth=1)

Your expected model summary:

```txt
==========================================================================================
Layer (type:depth-idx)                   Output Shape              Param #
==========================================================================================
FinalModel                               [1, 4]                    --
├─Sequential: 1-1                        [1, 1280, 7, 7]           2,223,872
├─Sequential: 1-2                        [1, 512]                  1,836,544
├─Linear: 1-3                            [1, 4]                    2,052
==========================================================================================
Total params: 4,062,468
Trainable params: 4,062,468
Non-trainable params: 0
```

<a name='4'></a>
## Train the Model

<a name='4.1'></a>
### 4.1 Prepare to Train the Model

You'll fit the model here, but first you'll set some of the parameters that go into fitting the model.

- EPOCHS: You'll train the model for 50 epochs
- BATCH_SIZE: Set the `BATCH_SIZE` to an appropriate value. You can look at the ungraded labs from this week for some examples.
- length_of_training_dataset: this is the number of training examples.  You can find this value by getting the length of `visualization_training_dataset`.
- length_of_validation_dataset: this is the number of validation examples.  You can find this value by getting the length of `visualization_validation_dataset`.
- steps_per_epoch: This is the number of steps it will take to process all of the training data.  
  - If the number of training examples is not evenly divisible by the batch size, there will be one last batch that is not the full batch size.
  - Try to calculate the number steps it would take to train all the full batches plus one more batch containing the remaining training examples. There are a couples ways you can calculate this.
    - You can use regular division `/` and import `math` to use `math.ceil()` [Python math module docs](https://docs.python.org/3/library/math.html)
    - Alternatively, you can use `//` for integer division, `%` to check for a remainder after integer division, and an `if` statement.

- validation_steps: This is the number of steps it will take to process all of the validation data.  You can use similar calculations that you did for the step_per_epoch, but for the validation dataset.

(A PyTorch `DataLoader` computes these numbers itself: `len(training_dataset)` is the number of batches. Calculating them by hand is still a good check.)

<a name='ex-06'></a>
### Exercise 6

In [ ]:
# You'll train 50 epochs
EPOCHS = 50

### START CODE HERE ###

# Choose a batch size
BATCH_SIZE = 64

# Get the length of the training set
length_of_training_dataset = len(visualization_training_dataset)

# Get the length of the validation set
length_of_validation_dataset = len(visualization_validation_dataset)

# Get the steps per epoch (may be a few lines of code).
# math.ceil rounds up, which accounts for the final partial batch.
steps_per_epoch = math.ceil(length_of_training_dataset / BATCH_SIZE)

# get the validation steps (per epoch) (may be a few lines of code)
validation_steps = length_of_validation_dataset//BATCH_SIZE
if length_of_validation_dataset % BATCH_SIZE > 0:
    validation_steps += 1

### END CODE HERE

# rebuild the data loaders with the batch size you chose
training_dataset = get_training_dataset(visualization_training_dataset)
validation_dataset = get_validation_dataset(visualization_validation_dataset)
print(f"steps per epoch: {steps_per_epoch} (DataLoader says {len(training_dataset)}), validation steps: {validation_steps} (DataLoader says {len(validation_dataset)})")

<a name='4.2'></a>
### 4.2 Fit the model to the data

In Keras `model.fit` ran the training loop for you. In PyTorch you write it yourself. The `evaluate` function below (the equivalent of `model.evaluate`) is provided: it runs the model over a data loader without computing gradients and returns the mean loss. Use it as a reference for the training loop, which additionally needs to:
- put the model in training mode (`model.train()`),
- zero the gradients (`optimizer.zero_grad()`), run the forward pass and compute the loss between the predicted and the true boxes,
- run the backward pass (`loss.backward()`) and update the weights (`optimizer.step()`).

If all goes well your model's training will start.

In [ ]:
def evaluate(model, loader, loss_fn, device):
    '''
    Runs the model over `loader` without computing gradients.

    Args:
      model (nn.Module) -- the bounding box regressor
      loader (DataLoader) -- yields (images, boxes) batches
      loss_fn (callable) -- loss applied to (predicted boxes, true boxes)
      device (torch.device) -- device the batches are moved to

    Returns:
      float -- mean loss over every batch of the loader
    '''
    model.eval()
    total_loss, count = 0.0, 0
    with torch.no_grad():
        for images, bboxes in loader:
            images, bboxes = images.to(device), bboxes.to(device)
            predictions = model(images)
            loss = loss_fn(predictions, bboxes)
            total_loss += loss.item() * len(images)
            count += len(images)
    return total_loss / count

<a name='ex-07'></a>
### Exercise 7

In [ ]:
history = {'loss': [], 'val_loss': []}

for epoch in range(EPOCHS):
    start = time.time()

    ### START CODE HERE ####

    # put the model in training mode
    model.train()

    # loop over the batches of `training_dataset`:
    #   move the images and boxes to the device, zero the gradients, compute the predictions and the loss,
    #   run the backward pass, take an optimizer step and accumulate the loss
    running_loss, seen = 0.0, 0
    for images, bboxes in training_dataset:
        images, bboxes = images.to(device), bboxes.to(device)   # shape: (N, 3, 224, 224), (N, 4)

        optimizer.zero_grad()                  # gradients accumulate, so clear the previous step
        predictions = model(images)            # shape: (N, 4)
        loss = loss_fn(predictions, bboxes)
        loss.backward()                        # fills .grad on every parameter
        optimizer.step()                       # applies the update

        running_loss += loss.item() * len(images)
        seen += len(images)

    train_loss = running_loss / seen

    ### END CODE HERE ###

    # evaluate on the validation set
    val_loss = evaluate(model, validation_dataset, loss_fn, device)

    history['loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    print(f"Epoch {epoch + 1}/{EPOCHS} - {time.time() - start:.0f}s - loss: {train_loss:.4f} - val_loss: {val_loss:.4f}")

<a name='5'></a>
## 5. Validate the Model

<a name='5-1'></a>
### 5.1 Loss

You can now evaluate your trained model's performance by checking its loss value on the validation set.

In [ ]:
loss = evaluate(model, validation_dataset, loss_fn, device)
print("Loss: ", loss)

<a name='5-2'></a>
### 5.2 Plot Loss Function

You can also plot the loss metrics.

In [ ]:
plot_metrics(history, "loss", "Bounding Box Loss", ylim=0.2)

<a name='5-3'></a>
### 5.3 Evaluate performance using IoU

You can see how well your model predicts bounding boxes on the validation set by calculating the Intersection-over-union (IoU) score for each image.

- You'll find the IoU calculation implemented for you.
- Predict on the validation set of images.
- Apply the `intersection_over_union` on these predicted bounding boxes.

In [ ]:
def intersection_over_union(pred_box, true_box):

    '''
    Computes the intersection over union of matched box pairs.

    Args:
      pred_box (array) -- predicted boxes, shape (N, 4) as (xmin, ymin, xmax, ymax)
      true_box (array) -- ground truth boxes, same shape and ordering

    Returns:
      array -- IoU per box, shape (N, 1)
    '''
    xmin_pred, ymin_pred, xmax_pred, ymax_pred =  np.split(pred_box, 4, axis = 1)
    xmin_true, ymin_true, xmax_true, ymax_true = np.split(true_box, 4, axis = 1)

    #Calculate coordinates of overlap area between boxes
    xmin_overlap = np.maximum(xmin_pred, xmin_true)
    xmax_overlap = np.minimum(xmax_pred, xmax_true)
    ymin_overlap = np.maximum(ymin_pred, ymin_true)
    ymax_overlap = np.minimum(ymax_pred, ymax_true)

    #Calculates area of true and predicted boxes
    pred_box_area = (xmax_pred - xmin_pred) * (ymax_pred - ymin_pred)
    true_box_area = (xmax_true - xmin_true) * (ymax_true - ymin_true)

    #Calculates overlap area and union area.
    overlap_area = np.maximum((xmax_overlap - xmin_overlap),0)  * np.maximum((ymax_overlap - ymin_overlap), 0)
    union_area = (pred_box_area + true_box_area) - overlap_area

    # Defines a smoothing factor to prevent division by 0
    smoothing_factor = 1e-10

    #Updates iou score
    iou = (overlap_area + smoothing_factor) / (union_area + smoothing_factor)

    return iou


def predict(model, images, device, batch_size=64):
    '''
    Runs the model over a numpy array of preprocessed images.

    Args:
      model (nn.Module) -- the bounding box regressor
      images (array) -- preprocessed images, shape (N, 3, 224, 224)
      device (torch.device) -- device the model runs on
      batch_size (int) -- how many images to push through at a time

    Returns:
      array -- predicted boxes, shape (N, 4)
    '''
    model.eval()
    outputs = []
    with torch.no_grad():
        for i in range(0, len(images), batch_size):
            batch = torch.from_numpy(images[i:i + batch_size]).to(device)
            outputs.append(model(batch).cpu())
    return torch.cat(outputs).numpy()

#Makes predictions
original_images, normalized_images, normalized_bboxes = dataset_to_numpy_with_original_bboxes_util(visualization_validation_dataset, N=500)
predicted_bboxes = predict(model, normalized_images.astype('float32'), device)


#Calculates IOU and reports true positives and false positives based on IOU threshold
iou = intersection_over_union(predicted_bboxes, normalized_bboxes)
iou_threshold = 0.5

print("Number of predictions where iou > threshold(%s): %s" % (iou_threshold, (iou >= iou_threshold).sum()))
print("Number of predictions where iou < threshold(%s): %s" % (iou_threshold, (iou < iou_threshold).sum()))

<a name='6'></a>
## 6. Visualize Predictions

Lastly, you'll plot the predicted and ground truth bounding boxes for a random set of images and visually see how well you did!

In [ ]:
n = 10
indexes = np.random.choice(len(predicted_bboxes), size=n)

iou_to_draw = iou[indexes]
norm_to_draw = original_images[indexes]
display_digits_with_boxes(original_images[indexes], predicted_bboxes[indexes], normalized_bboxes[indexes], iou[indexes], "True and Predicted values", bboxes_normalized=True)

## 7. Save the Model

Once you're satisfied with the results, you can save your model's weights. (In the original course the Keras model file `birds.keras` was uploaded to the Coursera grader; the PyTorch weights below are for your own use.)

In [ ]:
# Save the model you just trained
torch.save(model.state_dict(), "birds.pt")

**Congratulations on completing this assignment on predicting bounding boxes!**